# ಪಾಠ 10 - ಉತ್ಪಾದನೆಯಲ್ಲಿ AI ಏಜೆಂಟ್ಗಳು

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿ AI ಏಜೆಂಟ್ಗಳಿಗಾಗಿ **ಉತ್ಪಾದನಾ ಮಾದರಿಗಳನ್ನು** ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಚರ್ಚಿಸುವವು:

- **ನಿರೀಕ್ಷಣೆ** — ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯ ಮತ್ತು ಲಾಗಿಂಗ್ ಸೇರಿಸುವುದು
- **ಮೌಲ್ಯಮಾಪನ** — ಪ್ರತಿಕ್ರಿಯೆ ಗುಣಮಟ್ಟವನ್ನು ಅಂಕ ವಿಮರ್ಶಿಸುವ ಏಜೆಂಟ್ ಬಳಕೆ
- **ವೆಚ್ಚ ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಆಯ್ಕೆ ಮತ್ತು ಮಾದರಿ ಆಯ್ಕೆಗಾಗಿ ತಂತ್ರಗಳು

ದೃಶ್ಯ ಆಧಾರವು **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಬಳಕೆದಾರರಿಗೆ ಪ್ರವಾಸ ಯೋಜಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತದೆ, ಮೇಲ್ಭಾಗದಲ್ಲಿ ಮೇಲ್ವಿಚಾರಣೆ ಮತ್ತು ಮೌಲ್ಯಮಾಪನವನ್ನು ಜೋಡಿಸಿದೆ.


## ಸೆಟ್‌ಅಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಉತ್ಪಾದನಾ ಪರಿಗಣನೆಗಳು

ಪ್ರೋಟೋಟೈಪ್ಗಳಿಂದ ಉತ್ಪಾದನೆಯತ್ತ AI ಏಜೆಂಟುಗಳನ್ನು ಸಾಗಿಸುವುದು ಮೂರು ಸ್ತಂಭಗಳಿಗೆ ಸೂಕ್ಷ್ಮ ಗಮನವನ್ನು ಅಗತ್ಯವಿದೆ:

1. **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಏನು ಮಾಡುತ್ತಿದೆ, ಎಷ್ಟು ಸಮಯ ತೆಗೆದುಕೊಳ್ಳುತ್ತಿದೆ ಮತ್ತು ಯಾವ ಟೂಕಳನ್ನು ಕರೆಸುತ್ತಿದೆ ಎಂಬುದರ ವೀಕ್ಷಣೆ ಅಗತ್ಯ. ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಲಾಗಿಂಗ್ ಇಲ್ಲದೆ, ಉತ್ಪಾದನಾ ಸಮಸ್ಯೆಗಳ ಡಿಬಗ್ಗಿಂಗ್ ದರಿದ್ರ.

2. **ಮೌಲ್ಯಮಾಪನ** — ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಪರಿಶೀಲನೆಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಕಾಲಕಾಲಕ್ಕೆ ಸರಿಯಾದ, ಪೂರ್ಣಗೊಂಡ ಮತ್ತು ಸಹಾಯಕವಾಗಿರುವುದನ್ನು ಖಚಿತ ಪಡಿಸುತ್ತವೆ. ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ನಿರ್ದಿಷ್ಟ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಅಂಕುಡಿಕ್ಕುತ್ತದೆ.

3. ** ವೆಚ್ಚ ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಬಳಕೆ ನೇರವಾಗಿ ವೆಚ್ಚವನ್ನು ಪ್ರಭಾವಿಸುತ್ತದೆ. ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ, ಮಾದರಿ ಆಯ್ಕೆ ಮತ್ತು ಕ್ಯಾಶಿಂಗ್ ಮುಂತಾದ ತಂತ್ರಗಳು ಗುಣಮಟ್ಟಕ್ಕೆ ತಲ್ಲಣವಿಲ್ಲದೆ ಖರ್ಚುಗಳನ್ನು ನಿಯಂತ್ರಣದಲ್ಲಿ ಇಡಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


## ಅವಲೋಕನಕಾರಿಯಾದ ಏಜೆಂಟ್ ನಿರ್ಮಾಣ

ನಾವು ಪ್ರಯಾಣೋಪಕರಣಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ ಏಜೆಂಟ್ ಕರೆ ಸಮಯಾವಧಿಯೊಂದಿಗೆ ವರೆಹಿಸುತ್ತೇವೆ ώστε λ ಸಾವಧಾನತೆ ಪರಿಶೀಲಿಸಲು ಸಾಧ್ಯವಾಗುತ್ತದೆ. ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು OpenTelemetry ಅಥವಾ ಸಮಾನವಾದ ಟ್ರೇಸಿಂಗ್ ಬ್ಯಾಕೆಂಡ್‌ನೊಂದಿಗೆ ಏಕೀಕೃತಗೊಳಿಸುವಿರಿ.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ಖರ್ಚು ನಿರ್ವಹಣಾ వ్యూహಗಳು

ಉತ್ಪಾದನಾ AI ಪ್ರದೇಶಗಳಿಗೆ ಖರ್ಚನ್ನು ನಿಯಂತ್ರಿಸುವುದು ಅತ್ಯತكسيرವಾಗಿರುತ್ತದೆ. ಪ್ರಮುಖ వ్యూహಗಳು ಇಲ್ಲಿವೆ:

| వ్యూహ | వివರಣೆ |
|---|---|
| **ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ** | ವ್ಯವಸ್ಥೆಯ ಸೂಚನೆಗಳನ್ನು ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಇರಿಸಿ. ಇನ್‌ಪುಟ್ ಟೋಕನ್ಗಳನ್ನು ಕಡಿಮೆ ಮಾಡಲು ಅತಿರ redundantಂತರಿಂದ ಮುಕ್ತವಾಗಿ ಇರಿಸಿ. |
    "| **ಮಾಡೆಲ್ ಆಯ್ಕೆ** | ಸರಳ ಕಾರ್ಯಗಳಿಗೆ classification ಅಥವಾ extraction ಮುಂತಾದವುಗಳಿಗೆ ಸಣ್ಣ, ಕಡಿಮೆ ದುಬಾರಿ ಮಾದರಿಗಳನ್ನು (ಉದಾ. GPT-4.1-mini) ಬಳಸಿರಿ ಮತ್ತು ಸಂಕೀರ್ಣ ತರ್ಕಕ್ಕೆ ದೊಡ್ಡ ಮಾದರಿಗಳನ್ನು ಮೀಸಲಾಗಿಡಿ. |\n",
| **ಕೆಶಿಂಗ್** | ಒಪ್ಪಂದ ಸಾಧನೆಗಳ ಫಲಿತಾಂಶಗಳನ್ನು ಮತ್ತು ಇನ್ನಷ್ಟು ಪ್ರಶ್ನೆಗಳನ್ನು ಕೆಶ್ ಮಾಡಿ ಮುಂಭಾಗದ API ಕರೆಗಳನ್ನು ತಪ್ಪಿಸಿಕೊಳ್ಳಿ. |
| **ಟೋಕನ್ ಬಜೆಟ್‌ಗಳು** | ಅಪ್ರತ್ಯಾಶಿತವಾಗಿ ದೀರ್ಘ ಉತ್ತರಗಳನ್ನು ತಡೆಯಲು `max_tokens` ಮಿತಿಗಳನ್ನು ಹೊಂದಿಸಿ. |
| **ಬ್ಯಾಚಿಂಗ್** | ಸಾಧ್ಯವಾದಲ್ಲಿ, ಬಹಳಷ್ಟು ಬಳಕೆದಾರ ಪ್ರಶ್ನೆಗಳನ್ನು ಒಂದೇ API ಕರೆಗಾಗಿ ಗುಂಪು ಮಾಡಿ. |

ವಾಸ್ತವಿಕವಾಗಿ, ಒಂದು ಹಂತಗರ್ಭಿತವಾದ ವಿಧಾನ ಉತ್ತಮವಾಗಿ ಕೆಲಸ ಮಾಡುತ್ತದೆ: ಸರಳ ವಿನಂತಿಗಳನ್ನು ವೇಗವಾಗಿ, ಸಹಜವಾಗಿ ಖರ್ಚುಮಾಡಬಹುದಾದ ಮಾದರಿಗೇ ರವಾನಿಸಿ ಮತ್ತು ಕಠಿಣ ಪ್ರಶ್ನೆಗಳಿಗೆ ಮಾತ್ರ ಹೆಚ್ಚು ಸಾಮರ್ಥ್ಯ ಇರುವ ಮಾದರಿಗಳನ್ನು ಬಳಸಿ. 


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೀಗನ್ನು ಕಲಿತಿರಿ:

1. **ಟೈಮಿಂಗ್ ಮತ್ತು ಲಾಗಿಂಗ್** ಮೂಲಕ ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಅವಲೋಕನವನ್ನು ಸೇರಿಸಿ, ಅನುಸರಣೆ ಮತ್ತು ಮಾನಿಟರಿಂಗ್‌ಗಾಗಿ ಆಧಾರವನ್ನು ಹಾಕುವುದು.
2. ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯನ್ನು ಅಂಕಿತಗೊಳಿಸುವ ಇವ್ಯಾಲುವೇಟರ್ ಏಜೆಂಟ್ ಬಳಸಿ **ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳ ಮೌಲ್ಯಮಾಪನ** ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಮಾಡುವುದು.
3. ಪ್ರಾಂಪ್ಟ್ ಆಪ್ಟಿಮೈಸೇಷನ್, ಮಾದರಿ ಆಯ್ಕೆ, ಕ್ಯಾಶಿಂಗ್ ಮತ್ತು ಟೋಕನ್ ಬಜೆಟ್‌ಗಳ ಮೂಲಕ **ಖರ್ಚುಗಳನ್ನು ನಿರ್ವಹಿಸುವುದು**.

ಈ ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು ನಿಮ್ಮ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ಸ skaleyಲ್ಲಲ್ಲಿ ನಂಬಿಗಸ್ಥ, ಅಳೆಯಬಹುದಾದ ಮತ್ತು ವೆಚ್ಚ ಪರಿಣಾಮಕಾರಿಯಾಗಿ ಉಳಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
